## ✅ [Day3~4 - 최종 버전] 번개장터 실거래 데이터 기반 모델 학습

이 노트북은 Kaggle 데이터의 한계(`train_price_model.ipynb` 참고)를 
발견한 후, **번개장터 실시간 API 크롤링**으로 확보한 실거래 데이터 
(781건, 아이폰13~16·갤럭시S22~25·Z플립폴드3~5 등 30개 기종)로 
재구축한 최종 모델 학습 과정입니다.

브랜드별(Apple/Samsung) 분리 학습, 상태 등급(S/A/B/F) 체계 검증 등을 
포함하며, **이 노트북의 결과물이 실제 배포된 서비스(`server.py`)에서 
사용됩니다.**

In [14]:
##데이터 로드 및 확인
import pandas as pd
from phone_models import phone_models

df = pd.read_csv("data/bunjang_final_data.csv")
print("전체 데이터 개수:", len(df))
print(df.head())
print()
print("브랜드별 개수:")
print(df['brand'].value_counts())
print()
print("등급별 개수:")
print(df['condition'].value_counts())

전체 데이터 개수: 796
  search_keyword                                name   price storage  \
0         아이폰 13              애플 아이폰 13 미니 화이트 128GB  250000   128GB   
1         아이폰 13  아이폰13미니 미드나잇 128GB 배터리100 춘천 강릉 제천  430000   128GB   
2         아이폰 13                 아이폰 13 128GB 판매합니다.  265000   128GB   
3         아이폰 13             아이폰13프로 512기가 780232 28  370000   512기가   
4         아이폰 13                  애플 아이폰 13 미니 128GB  260000   128GB   

   update_time  used_flag  storage_gb condition  brand  
0   1784779578          1         128         B  Apple  
1   1784779289          1         128         B  Apple  
2   1784777713          1         128         B  Apple  
3   1784777826          1         512         B  Apple  
4   1784776680          1         128         B  Apple  

브랜드별 개수:
brand
Samsung    551
Apple      245
Name: count, dtype: int64

등급별 개수:
condition
B    621
S    110
A     59
F      6
Name: count, dtype: int64


In [15]:
##매핑 확인
def normalize_key(keyword):
    return keyword.replace(" ", "")

df['model_key'] = df['search_keyword'].apply(normalize_key)

model_keys = set(phone_models.keys())
data_keys = set(df['model_key'].unique())

print("매핑 안 되는 키 확인:")
print(data_keys - model_keys)

매핑 안 되는 키 확인:
set()


In [16]:
## 각 모델이 실제로 존재하는 저장용량이 아니면 제외 (521GB, 32GB 등 오염된 값 제거)
def is_valid_storage(row):
    model_key = row['model_key']
    storage = row['storage_gb']
    spec = phone_models.get(model_key, {})
    valid_storages = spec.get('storage_options', {}).keys()
    return storage in valid_storages

before_count = len(df)
df = df[df.apply(is_valid_storage, axis=1)]
after_count = len(df)
print(f"존재하지 않는 저장용량 제거: {before_count}개 → {after_count}개")

존재하지 않는 저장용량 제거: 796개 → 722개


In [17]:
##"미니" 모델 제외
before_count = len(df)
df = df[~df['name'].str.contains('미니|mini|Mini', na=False)]
after_count = len(df)
print(f"미니 모델 제외: {before_count}개 → {after_count}개")
print(df['search_keyword'].value_counts())

미니 모델 제외: 722개 → 700개
search_keyword
갤럭시 S24 플러스    62
갤럭시 S25 플러스    47
갤럭시 Z플립4       45
갤럭시 S24        43
갤럭시 S23        39
갤럭시 S22        34
갤럭시 Z플립5       34
아이폰 15         32
갤럭시 Z플립3       32
갤럭시 Z폴드4       32
아이폰 13 프로맥스    25
갤럭시 S22 울트라    25
아이폰 14         23
갤럭시 S23 플러스    22
아이폰 15 프로맥스    21
갤럭시 S25        21
아이폰 14 프로맥스    19
아이폰 13 프로      18
갤럭시 Z폴드3       17
갤럭시 Z폴드5       17
갤럭시 S25 울트라    16
아이폰 16         14
갤럭시 S23 울트라    14
아이폰 16 프로맥스    12
갤럭시 S22 플러스    10
아이폰 16 프로       8
아이폰 13          7
아이폰 15 프로       6
아이폰 14 프로       4
갤럭시 S24 울트라     1
Name: count, dtype: int64


In [18]:
##매핑표에서 스펙 정보 결합
def get_spec_value(row, key):
    model_key = row['model_key']
    spec = phone_models.get(model_key, {})
    return spec.get(key)

def get_new_price(row):
    model_key = row['model_key']
    storage = row['storage_gb']
    spec = phone_models.get(model_key, {})
    storage_options = spec.get('storage_options', {})

    if storage in storage_options:
        return storage_options[storage]
    elif len(storage_options) > 0:
        closest = min(storage_options.keys(), key=lambda x: abs(x - storage))
        return storage_options[closest]
    else:
        return None

df['screen_size'] = df.apply(lambda row: get_spec_value(row, 'screen_size'), axis=1)
df['ram'] = df.apply(lambda row: get_spec_value(row, 'ram'), axis=1)
df['battery'] = df.apply(lambda row: get_spec_value(row, 'battery'), axis=1)
df['weight'] = df.apply(lambda row: get_spec_value(row, 'weight'), axis=1)
df['release_year'] = df.apply(lambda row: get_spec_value(row, 'release_year'), axis=1)
df['new_price'] = df.apply(get_new_price, axis=1)

print(df[['search_keyword', 'storage_gb', 'new_price', 'screen_size', 'ram', 'battery', 'price']].head(10))
print()
print("결측치 확인:")
print(df[['screen_size', 'ram', 'battery', 'weight', 'release_year', 'new_price']].isnull().sum())

   search_keyword  storage_gb  new_price  screen_size  ram  battery   price
2          아이폰 13         128    1078000          6.1    4     3227  265000
3          아이폰 13         512    1485000          6.1    4     3227  370000
14         아이폰 13         256    1221000          6.1    4     3227  390000
15         아이폰 13         128    1078000          6.1    4     3227  430000
21         아이폰 13         256    1221000          6.1    4     3227  400000
23         아이폰 13         128    1078000          6.1    4     3227  205000
25         아이폰 13         128    1078000          6.1    4     3227  380000
29      아이폰 13 프로         128    1342000          6.1    6     3095  440000
30      아이폰 13 프로         256    1474000          6.1    6     3095  550000
31      아이폰 13 프로         128    1342000          6.1    6     3095  375000

결측치 확인:
screen_size     0
ram             0
battery         0
weight          0
release_year    0
new_price       0
dtype: int64


In [19]:
##원-핫 인코딩
model_df = df[['brand', 'condition', 'screen_size', 'ram', 'battery', 'weight',
               'release_year', 'storage_gb', 'new_price', 'price']].copy()

model_df_encoded = pd.get_dummies(model_df, columns=['brand', 'condition'], drop_first=True)

print(model_df_encoded.shape)
print(model_df_encoded.columns.tolist())

(700, 12)
['screen_size', 'ram', 'battery', 'weight', 'release_year', 'storage_gb', 'new_price', 'price', 'brand_Samsung', 'condition_B', 'condition_F', 'condition_S']


In [20]:
##통합 모델 학습 (비교용)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X = model_df_encoded.drop(columns=['price'])
y = model_df_encoded['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"통합 모델 MAE: {mae:,.0f}원")
print(f"통합 모델 R²: {r2:.3f}")

통합 모델 MAE: 85,078원
통합 모델 R²: 0.818


In [21]:
##브랜드별 분리 학습 + 저장
import joblib

# Apple 모델
apple_df = model_df_encoded[model_df_encoded['brand_Samsung'] == 0].drop(columns=['brand_Samsung'])
X_apple = apple_df.drop(columns=['price'])
y_apple = apple_df['price']
X_tr, X_te, y_tr, y_te = train_test_split(X_apple, y_apple, test_size=0.2, random_state=42)

apple_model = RandomForestRegressor(n_estimators=100, random_state=42)
apple_model.fit(X_tr, y_tr)
print("Apple 모델 R²:", r2_score(y_te, apple_model.predict(X_te)))

joblib.dump(apple_model, "apple_price_model.joblib")
joblib.dump(X_apple.columns.tolist(), "apple_model_columns.joblib")

# Samsung 모델
samsung_df = model_df_encoded[model_df_encoded['brand_Samsung'] == 1].drop(columns=['brand_Samsung'])
X_samsung = samsung_df.drop(columns=['price'])
y_samsung = samsung_df['price']
X_tr, X_te, y_tr, y_te = train_test_split(X_samsung, y_samsung, test_size=0.2, random_state=42)

samsung_model = RandomForestRegressor(n_estimators=100, random_state=42)
samsung_model.fit(X_tr, y_tr)
print("Samsung 모델 R²:", r2_score(y_te, samsung_model.predict(X_te)))

joblib.dump(samsung_model, "samsung_price_model.joblib")
joblib.dump(X_samsung.columns.tolist(), "samsung_model_columns.joblib")

Apple 모델 R²: 0.6197473109989728
Samsung 모델 R²: 0.7114071498890109


['samsung_model_columns.joblib']

In [22]:
##피처 중요도 확인
print("=== Apple 모델 피처 중요도 ===")
apple_feat_imp = pd.Series(apple_model.feature_importances_, index=X_apple.columns).sort_values(ascending=False)
print(apple_feat_imp)

print()
print("=== Samsung 모델 피처 중요도 ===")
samsung_feat_imp = pd.Series(samsung_model.feature_importances_, index=X_samsung.columns).sort_values(ascending=False)
print(samsung_feat_imp)

=== Apple 모델 피처 중요도 ===
battery         0.301565
ram             0.190430
new_price       0.175337
release_year    0.137721
screen_size     0.066688
weight          0.061899
condition_S     0.024603
storage_gb      0.023247
condition_B     0.018511
condition_F     0.000000
dtype: float64

=== Samsung 모델 피처 중요도 ===
release_year    0.617782
battery         0.250768
new_price       0.044839
weight          0.029719
screen_size     0.023053
storage_gb      0.011813
condition_S     0.007890
condition_B     0.007680
condition_F     0.004404
ram             0.002051
dtype: float64


In [23]:
##등급별 신품가 대비 비율 (참고용)
df['real_ratio'] = df['price'] / df['new_price']
print(df.groupby('condition')['real_ratio'].agg(['mean', 'count']))

               mean  count
condition                 
A          0.419739     50
B          0.355771    542
F          0.105079      4
S          0.373146    104


In [24]:
## 모델 x 저장용량별 표본 개수 확인
storage_counts = df.groupby(['search_keyword', 'storage_gb']).size().reset_index(name='count')
storage_counts_sorted = storage_counts.sort_values('count')

print("표본이 5개 이하인 (모델, 용량) 조합:")
print(storage_counts_sorted[storage_counts_sorted['count'] <= 5])

print()
print("전체 통계:")
print(storage_counts['count'].describe())

표본이 5개 이하인 (모델, 용량) 조합:
   search_keyword  storage_gb  count
12    갤럭시 S24 울트라         256      1
49      아이폰 15 프로         128      1
51      아이폰 15 프로         512      1
59      아이폰 16 프로         512      1
48         아이폰 15         512      1
32         아이폰 13         512      1
40         아이폰 14         512      1
16        갤럭시 S25         512      2
37    아이폰 13 프로맥스         512      2
41      아이폰 14 프로         128      2
31         아이폰 13         256      2
45    아이폰 14 프로맥스         512      2
56         아이폰 16         512      2
57      아이폰 16 프로         128      2
42      아이폰 14 프로         256      2
9     갤럭시 S23 플러스         512      3
54         아이폰 16         128      3
30         아이폰 13         128      4
50      아이폰 15 프로         256      4
61    아이폰 16 프로맥스         512      4
58      아이폰 16 프로         256      5
24       갤럭시 Z폴드4         512      5
2     갤럭시 S22 울트라         512      5
7     갤럭시 S23 울트라         512      5
53    아이폰 15 프로맥스         512      5

전체 통계:
count 